# PGR107 Python Programming | Home Exam 2026
**Bachelor of Data Science, Kristiania University of Applied Sciences**

**Candidates:** xxx & xxx

## How to use
Run the last coding cell to access the menu, which presents our exam submission.


## Imports

In [286]:
import numpy as np

## Code

In [287]:
def printMenu():
    print()
    print("Which question do you want to run?")
    print("-----------------------------------")
    print("0. Exit")
    print("1. Question 1 - Supernacci")
    print("2. Question 2 - Dice Game")
    print("3. Question 3 - Coordinates")
    print("4. Question 4 - Matrix Trace")
    print("-----------------------------------")
    print()

def pressEnterToContinue():
    try:
        print("\nClick OK to continue... (or enter if in terminal)")
        input()
    except KeyboardInterrupt:
        pass

def menu():
    choice = ' '

    while choice not in ('0'):
        printMenu()

        try:
            choice = input("Enter a number: ")
        except KeyboardInterrupt:
            print("\nExiting...")
            break

        if choice == '1':
            question1()
            pressEnterToContinue()
            choice = ' '
        elif choice == '2':
            question2()
            pressEnterToContinue()
            choice = ' '
        elif choice == '3':
            question3()
            pressEnterToContinue()
            choice = ' '
        elif choice == '4':
            question4()
            pressEnterToContinue()
            choice = ' '
        elif choice == '0':
            print("Bye")
        else:
            print("Invalid choice")

---
## Question 1: Supernacci Numbers

The Supernacci numbers `s(n)` are defined as:
- `s(1) = 1`
- `s(n)` is the sum of the last `(n - 1) % 3` Supernacci numbers, or `s(n-1)` if that sum would be empty (i.e. when `(n-1) % 3 == 0`).

**Strategy:** Build the sequence iteratively in a list. For each new term, we use slicing `s[-j:]` and the built-in `sum()` to compute the value.

In [288]:
def question1():
    def supernacci(n):
        s = [1]
        for i in range(2, n + 1):
            j = (i - 1) % 3
            # When (n-1) % 3 == 0, the slice s[-0:] would mean s[0:], which is the entire list, not an empty one.
            if j == 0:
                # Sum would be empty, use previous term instead
                s.append(s[-1])
            else:
                # Sum of the last j elements using slicing
                s.append(sum(s[-j:]))

        return s[n - 1]

    # Output the first 12 Supernacci numbers to verify the pattern
    print("n  | s(n)")
    print("-----------------------------------")
    for n in range(1, 13):
        print(f"{n:<3}| {supernacci(n)}")

---
## Question 2: Gambling Program

Fixing a bugged implementation of a gambling program.

**Bugs fixed:**
- Typos, all "guesss" instances changed to "guess"
- In getDiceGuess() the return statement was indented inside the while loop
- In getDiceGuess() trailing spaces in `'1 '` etc., user input would never match
- In getDiceGuess() returns a string, but dice is an int. Added `int()` conversion
- In rollDice() `random.randint` changed to `rnd.randint`
- In rollDice() `randint(1, 5)` changed to `randint(1, 4)`
- In rollDice() `if dice = guess:` declaration changed to comparison `==`
- In rollDice() Last `if dice = guess` missing the colon :
- None of the functions get called, added function calls

**Architecture changes:**
- Added checkDice function instead of the last `if` chain. It was unnecessarily indented and
  generally bad practice since it is inconvenient to both read and maintain. Added `maxAttempts`
  variable to facilitate this function.
- Removed "global" variables `dice` and `guess`. Since our implementation uses a menu wrapper,
  they aren't technically global, but nonlocal. Regardless the program becomes more robust
  by confining variables within the scope they are required.


In [289]:
import random as rnd

def question2():
    def getDiceGuess():
        guessRaw = ''
        while guessRaw not in ('1', '2', '3', '4'):
            print('Guess the number on a 4-sided dice! Enter a number between 1 and 4:')
            guessRaw = input()
        return int(guessRaw)

    def checkDice(dice, guess, attempts, maxAttempts):
        if dice == guess:
            print('Correct! Number of attempts:', attempts)
            return True
        else:
            print('Wrong! Remaining attempts:', maxAttempts - attempts)
            return False

    def rollDice():
        roll = rnd.randint(1, 4)
        return roll

    dice = rollDice()
    maxAttempts = 4

    for attempts in range(1, maxAttempts + 1): # lower boundary inclusive, upper exclusive
        guess = getDiceGuess()
        if checkDice(dice, guess, attempts, maxAttempts):
            return

    print('Game over! The number was', dice)

---
## Question 3:

Use the following approximations:

`1 degree of latitude = 111 km`

`1 degree of longitude = 68 km`

Assume that each city has a square shape and that the city size can be added from the center in each direction, as you can see in the image. Ignore the curvature of the Earth when working with latitude and longitude.

a) (6 points) Find the nearest city and POI
 - Function that returns the closest POI for a given pair of coordinates,
 - Function that returns the closest city for a given pair of coordinates.
   Then ask the user to enter a latitude and longitude, and print the closest city and closest POI.

b) (6 points) Compute city boundaries
 - Extend the city data by computing the city boundaries: north, south, east, west.

c) (6 points) Assign POIs to cities
For each city, determine which POIs are located inside its boundaries.
A POI belongs to a city if:
 - its latitude lies between the south and north boundaries
 - its longitude lies between the west and east boundaries

d) (6 points) Compute average POI rating per city
For each city, compute the average customer rating of all POIs located in that city.

e) (6 points) Order POIs by distance from the city center
For each city, create a list of POI names ordered by their distance from the city center, from nearest to farthest.

In [290]:
def question3():

    latitudeMultiplier = 111
    longitudeMultiplier = 68

    cities = {'Bergen':    [60.364098, 5.404422,  15],
              'Oslo':      [59.926959, 10.779454, 10],
              'Trondheim': [63.433654, 10.390277, 5]
    }

    poi = {(63.426806, 10.396712): ['Nidarosdommen',     4.5],
           (59.926699, 10.701075): ['Vigelandsparken',   5],
           (59.907704, 10.753163): ['Operahuset',        4],
           (60.397561, 5.322833):  ['Bryggen',           4.5],
           (60.387639, 5.321623):  ['Universitetmuseet', 5],
           (60.342901, 5.336796):  ['Gamlehaugen',       4]
    }

    # Task a: Find the nearest city and POI
    def returnClosestPoi(userCoords):
        shortestDistance = np.inf
        closestPoi       = None
        closestCoords    = None

        print("\n--| Searching POIs |--")
        for coords, data in poi.items():
            poiLatitude  = coords[0]
            poiLongitude = coords[1]
            print(f"  Testing POI: {data[0]:<20} at ({poiLatitude}, {poiLongitude})")

            latitudeDistance  = abs(userCoords[0] - poiLatitude)  * latitudeMultiplier
            longitudeDistance = abs(userCoords[1] - poiLongitude) * longitudeMultiplier
            distance = latitudeDistance + longitudeDistance

            if distance < shortestDistance:
                shortestDistance = distance
                closestPoi       = data
                closestCoords    = coords

        return closestPoi, closestCoords

    def returnClosestCity(userCoords):
        shortestDistance = np.inf
        closestCity      = None
        closestCoords    = None

        print("\n--| Searching Cities |--")
        for city, values in cities.items():
            cityLatitude  = values['center'][0]
            cityLongitude = values['center'][1]
            print(f"  Testing City: {city:<12} at ({cityLatitude}, {cityLongitude})")

            latitudeDistance  = abs(userCoords[0] - cityLatitude)  * latitudeMultiplier
            longitudeDistance = abs(userCoords[1] - cityLongitude) * longitudeMultiplier
            distance = latitudeDistance + longitudeDistance

            if distance < shortestDistance:
                shortestDistance = distance
                closestCity      = city
                closestCoords    = [cityLatitude, cityLongitude]

        return closestCity, closestCoords

    def getUserCoords():
        try:
            while True:
                try:
                    userLatitude = float(input("Enter your latitude: "))
                    break
                except ValueError:
                    print("Invalid input.")

            while True:
                try:
                    userLongitude = float(input("Enter your longitude: "))
                    break
                except ValueError:
                    print("Invalid input.")

            return [userLatitude, userLongitude]

        except KeyboardInterrupt:
            print("\nExiting...")
            return None

    # Task b: Extend cities with boundaries
    def setCityBoundaries():
        for city, values in cities.items():
            lat, lon, size = values[0], values[1], values[2]
            cities[city] = {
                'center': [lat, lon],
                'size':   size,
                'north':  lat + (size / latitudeMultiplier),
                'south':  lat - (size / latitudeMultiplier),
                'east':   lon + (size / longitudeMultiplier),
                'west':   lon - (size / longitudeMultiplier)
            }

    # Task c: Assign POIs to cities
    def setCityPois():
        for city in cities:
            cities[city]['pois'] = []

        for coords, data in poi.items():
            poiLat, poiLon = coords[0], coords[1]

            for city, values in cities.items():
                insideLat = values['south'] <= poiLat <= values['north']
                insideLon = values['west']  <= poiLon <= values['east']

                if insideLat and insideLon:
                    values['pois'].append(data)

    # Task d: Compute average POI rating per city
    def computeAvgRatings():
        for city, values in cities.items():
            ratings = [p[1] for p in values['pois']]
            values['avgRating'] = sum(ratings) / len(ratings)

    # Task e: Order POIs by distance from city center
    def sortPoisByDistance():
        for city, values in cities.items():
            cityLat = values['center'][0]
            cityLon = values['center'][1]

            distances = []
            for coords, data in poi.items():
                if data in values['pois']:
                    latDist  = abs(cityLat - coords[0]) * latitudeMultiplier
                    lonDist  = abs(cityLon - coords[1]) * longitudeMultiplier
                    distances.append((latDist + lonDist, data[0]))

            distances.sort()
            values['poisByDistance'] = distances

    # Main flow
    print("Find the closest POI or city based on your coordinates")
    userCoords = getUserCoords()

    if userCoords is None:
            return

    setCityBoundaries()
    setCityPois()
    computeAvgRatings()
    sortPoisByDistance()

    closestPoi,  poiCoords  = returnClosestPoi(userCoords)
    closestCity, cityCoords = returnClosestCity(userCoords)

    print("\n--| Results |--")
    print(f"  {'User coordinates:':<18} {'':<20}  ({userCoords[0]}, {userCoords[1]})")
    print(f"  {'Closest POI:':<18} {closestPoi[0]:<20}  ({poiCoords[0]}, {poiCoords[1]})")
    print(f"  {'Closest City:':<18} {closestCity:<20}  ({cityCoords[0]}, {cityCoords[1]})")

    print("\n--| City Boundaries |--")
    for city, values in cities.items():
        print(f"  {city} (center: {values['center']}, size: {values['size']} km)")
        print(f"    north: {values['north']:.6f}    south: {values['south']:.6f}")
        print(f"    east:  {values['east']:9.6f}    west:  {values['west']:9.6f}")

    print("\n--| POIs per City |--")
    for city, values in cities.items():
        print(f"  {city:<12} {[f'{p[0]} ({p[1]})' for p in values['pois']]}")
    print()

    print("\n--| Average POI Rating per City |--")
    for city, values in cities.items():
        print(f"  {city:<12} avg rating: {values['avgRating']:.2f}")

    print("\n--| POIs by Distance from City Center |--")
    for city, values in cities.items():
        print(f"  {city}")
        for distance, name in values['poisByDistance']:
            print(f"    {name:<20} {distance:.2f} km")

    print()

---
## Question 4: Matrix Trace

Given a rank 2 numpy array A (a matrix), check whether the number of columns equals the number of rows. If (and only if) they coincide, compute and output the sum of elements on the main diagonal.

**Strategy:** First implement manually by iterating over the diagonal indices `A[i, i]`. Then verify the result using NumPy's built-in `np.trace()` to confirm correctness.

In [316]:
def question4():
    def trace(A):
        # check whether the number of columns equals the number of rows
        if A.shape[0] != A.shape[1]:
            print("Invalid matrix, trace operation aborted.")
            return

        # Manual implementation: sum diagonal elements A[i, i]
        total = 0
        for i in range(A.shape[0]):
            total += A[i, i]

        print(f"Manual trace:    {total}")

        # np.trace() used to verify our manual result is correct
        print(f"np.trace():      {np.trace(A)}\n")

    A = np.array([[1, 2, 3],
                  [4, 5, 6],
                  [7, 8, 9]])
    B = np.array([[3, 1, 7],
                  [7, 1, 4],
                  [2, 8, 5]])
    C = np.array([[1, 2, 3],
                  [4, 5, 6]])


    print(f"Matrix A:\n{A}")
    trace(A)
    print(f"Matrix B:\n{B}")
    trace(B)
    print(f"Matrix C:\n{C}")
    trace(C)

---
## Question 5: Use-case study

https://www.udir.no/tall-og-forskning/statistikk/statistikk-videregaende-skole/karakterer-vgs/


### 5.1 — Dataset Selection and Loading

**Subject:** Programfag Elektro og datateknologi
We chose this subject because it is a traditionally male-dominated field, making it well-suited for studying gender representation and performance over time.

**Assessment type:** Standpunkt
We limit to Standpunkt to get a consistent, comparable baseline across years. Standpunkt is available for all students regardless of whether they sit an exam, giving us the most complete dataset.

**School area:** Oslo (selected from 'Enhet')

**Years:** 2020-21, 2021-22, 2022-23, 2023-24, 2024-25
We download one CSV per school year to capture multi-year trends. Each file is loaded individually and merged into a single DataFrame with a year column added.

---

### 5.2 — Data visualization and takeaways

---

### 5.3 — Problem Definition and Analysis

**Research question:**
In the subject Elektro og datateknologi at Oslo high schools from 2020-2025, how does gender representation and academic performance compare between male and female students?

**Hypotheses:**
- **H1 (Representation):** Females are underrepresented relative to males, but the gap has been gradually closing from 2020 to 2025.
- **H2 (Performance):** Despite being underrepresented, female students perform at least as well as or better than male students in terms of Standpunkt grades, consistent with the broader finding that girls outperform boys academically (Moran, 2024).

**Implementation:**
- Group data by year and gender to test H1
- Compute average grades by gender per year to test H2
- Visualize both trends and draw conclusions

---

### Sources

Moran, R. (2024, January 9). *Girls outperform boys from primary school to university.* Cambridge University Press & Assessment. https://www.cambridge.org/news-and-insights/girls-outperform-boys

Utdanningsdirektoratet. (2025). *Karakterstatistikk for videregående skole.* https://www.udir.no/tall-og-forskning/statistikk/statistikk-videregaende-skole/karakterer-vgs/

In [317]:
menu()


Which question do you want to run?
-----------------------------------
0. Exit
1. Question 1 - Supernacci
2. Question 2 - Dice Game
3. Question 3 - Coordinates
4. Question 4 - Matrix Trace
-----------------------------------

Matrix A:
[[1 2 3]
 [4 5 6]
 [7 8 9]]
Manual trace:    15
np.trace():      15

Matrix B:
[[3 1 7]
 [7 1 4]
 [2 8 5]]
Manual trace:    9
np.trace():      9

Matrix C:
[[1 2 3]
 [4 5 6]]
Invalid matrix, trace operation aborted.

Click OK to continue... (or enter if in terminal)

Which question do you want to run?
-----------------------------------
0. Exit
1. Question 1 - Supernacci
2. Question 2 - Dice Game
3. Question 3 - Coordinates
4. Question 4 - Matrix Trace
-----------------------------------


Exiting...
